# Figura 8 · Calcula tu caso

Este cuaderno crea la Figura 8 del trabajo con **Plotly**: una calculadora con un **deslizador de sueldo** y un **selector de comunidad autónoma**. El resultado es un archivo `figura8.html` que se coloca junto a `index.html` y se incrusta en la web con un iframe.

**Qué hace.** Se elige la comunidad y el sueldo bruto de 2024 (de 10.000 a 100.000 €, de 1.000 en 1.000 €). La calculadora:

1. Lleva ese sueldo **hacia atrás con el IPC de la comunidad**: se supone que el sueldo de cada año valía lo mismo que el de 2024 en poder de compra. Por ejemplo, 30.000 € de 2024 equivalen a unos 24.250 € de 2016 con el IPC nacional.
2. Calcula el IRPF de cada año de **2015 a 2024** con la **tarifa real** y con la **tarifa actualizada** con el IPC de la comunidad desde 2016.
3. Muestra a la izquierda la nómina de 2024 con las dos tarifas y a la derecha una **tabla año a año con el total**.

**Datos.** El IPC anual por comunidad, `data/processed/ipc_anual_ccaa.csv`, y el modelo fiscal del repositorio (`src/modelo_fiscal.py`). No se crea ninguna tabla nueva.

**Supuestos** (los del modelo fiscal): asalariado soltero, sin hijos ni otras rentas. **La escala autonómica se supone igual a la estatal en todas las comunidades**: el selector cambia el IPC con el que se calcula, no el impuesto autonómico, porque no tenemos las escalas de cada comunidad. El modelo empieza en 2015, por eso la tabla no llega a 2008. **No incluye el País Vasco ni Navarra**, que tienen su propio IRPF.

## 1. Preparación

In [1]:
import os, subprocess, sys
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO = "https://github.com/sortaria24-max/08MBID_TrabajoPractico.git"
if not os.path.exists("src/modelo_fiscal.py"):
    if os.path.exists("../src/modelo_fiscal.py"):   # ejecutado desde la carpeta notebooks/
        os.chdir("..")
    else:                                            # Colab: descargamos el repositorio
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
        os.chdir("08MBID_TrabajoPractico")
sys.path.insert(0, "src")
import modelo_fiscal as mf

AZUL, NARANJA, TINTA, TINTA2, GRIS = "#2A78D6", "#EB6834", "#14130F", "#55534C", "#D9D6CC"
COLOR = {"neto": "#9DC1EC", "ss": "#C9C5B8", "irpf": NARANJA}
AÑOS = list(range(2015, 2025))
SUELDOS = list(range(10000, 100001, 1000))

def euros(x):
    return f"{x:,.0f}".replace(",", ".") + " €"

def pct(x):
    return f"{x:.1f}".replace(".", ",") + " %"

## 2. El IPC de cada comunidad

Usamos el índice medio anual de cada comunidad. Quitamos el País Vasco y Navarra, que tienen su propio IRPF, y ponemos «Nacional» la primera.

In [2]:
ipc = pd.read_csv("data/processed/ipc_anual_ccaa.csv")
ipc = ipc.pivot(index="ccaa", columns="anio", values="ipc")
ipc = ipc.drop(index=["País Vasco", "Navarra, Comunidad Foral de"])
ipc = ipc.loc[["Nacional"] + sorted(i for i in ipc.index if i != "Nacional")]
NOMBRE_CCAA = {"Nacional": "Media nacional", "Asturias, Principado de": "Asturias", "Balears, Illes": "Illes Balears",
               "Madrid, Comunidad de": "Comunidad de Madrid", "Murcia, Región de": "Región de Murcia",
               "Rioja, La": "La Rioja", "Castilla - La Mancha": "Castilla-La Mancha"}
print(len(ipc), "opciones:", [NOMBRE_CCAA.get(c, c) for c in ipc.index])

16 opciones: ['Media nacional', 'Andalucía', 'Aragón', 'Asturias', 'Illes Balears', 'Canarias', 'Cantabria', 'Castilla-La Mancha', 'Castilla y León', 'Cataluña', 'Comunitat Valenciana', 'Extremadura', 'Galicia', 'Comunidad de Madrid', 'Región de Murcia', 'La Rioja']


## 3. El cálculo, año a año

Para una comunidad `c` y un sueldo `S` de 2024:

- **Sueldo de cada año:** `S × IPC(año) / IPC(2024)`.
- **IRPF con la tarifa real:** el modelo fiscal con la tarifa de ese año.
- **IRPF con la tarifa actualizada:** el modelo fiscal con los umbrales, mínimos y gastos multiplicados por `IPC(año) / IPC(2016)` (en 2015 y 2016 coinciden las dos).

Comprobamos que con el IPC nacional el resultado de 2024 coincide con el de la tabla `curva_bruto_neto.csv` del cuaderno principal.

In [3]:
def calculo(ccaa, s2024):
    p = ipc.loc[ccaa]
    filas = []
    for y in AÑOS:
        bruto = s2024 * p[y] / p[2024]
        f = 1.0 if y < mf.AÑO_BASE_INDEXACION else p[y] / p[mf.AÑO_BASE_INDEXACION]
        real, ss = mf.irpf_individual(bruto, y, 0.0, 1.0)
        act, _ = mf.irpf_individual(bruto, y, 0.0, f)
        filas.append(dict(anio=y, bruto=bruto, ss=ss, real=real, actualizada=act))
    return pd.DataFrame(filas)

curva = pd.read_csv("data/processed/curva_bruto_neto.csv")
for s in (24000, 30000, 50000):
    d = calculo("Nacional", s).iloc[-1]
    ref = curva[(curva.anio == 2024) & (curva.bruto == s)].iloc[0]
    assert abs(d.real - ref.irpf) < 0.01 and abs(d.actualizada - ref.irpf_indexado) < 1, s
calculo("Nacional", 30000).round(0)

,anio,bruto,ss,real,actualizada
0,2015,24298.0,1543.0,3338.0,3338.0
1,2016,24249.0,1540.0,3324.0,3324.0
2,2017,24723.0,1570.0,3457.0,3389.0
3,2018,25137.0,1596.0,3573.0,3445.0
4,2019,25313.0,1607.0,3623.0,3470.0
5,2020,25231.0,1602.0,3600.0,3458.0
6,2021,26012.0,1652.0,3819.0,3565.0
7,2022,28194.0,1790.0,4432.0,3864.0
8,2023,29190.0,1883.0,4703.0,3992.0
9,2024,30000.0,1941.0,4929.0,4101.0


## 4. La calculadora

- **Izquierda:** la nómina de 2024 con las dos tarifas (lo que llega a la cuenta, la Seguridad Social y el IRPF).
- **Derecha:** la tabla año a año, con el sueldo equivalente de cada año, el IRPF con las dos tarifas y la fila de **total 2015-2024**.
- **Selector de comunidad** (arriba) y **deslizador de sueldo** (abajo).

Como la web no usa JavaScript propio, Plotly guarda ya calculadas todas las combinaciones: cada comunidad tiene su propio juego de posiciones del deslizador, y al elegir otra comunidad el selector carga ese juego (el sueldo vuelve a 24.000 €).

In [4]:
FILAS = ["Tarifa<br>actualizada", "Tarifa<br>real"]
INICIAL = SUELDOS.index(24000)

def estado(ccaa, s2024):
    d = calculo(ccaa, s2024)
    u = d.iloc[-1]
    neto, neto_act = u.bruto - u.ss - u.real, u.bruto - u.ss - u.actualizada
    partes = {"neto": [neto_act, neto], "ss": [u.ss, u.ss], "irpf": [u.actualizada, u.real]}
    partes = {k: [round(v) for v in vs] for k, vs in partes.items()}   # euros enteros: el archivo pesa menos
    textos = {k: [euros(v) if v >= 7000 else "" for v in vs] for k, vs in partes.items()}
    celdas = [[str(a) for a in d.anio] + ["<b>Total</b>"],
              [euros(v) for v in d.bruto] + [""],
              [euros(v) for v in d.real] + [f"<b>{euros(d.real.sum())}</b>"],
              [euros(v) for v in d.actualizada] + [f"<b>{euros(d.actualizada.sum())}</b>"]]
    datos = {"x": [partes["neto"], partes["ss"], partes["irpf"], None],
             "text": [textos["neto"], textos["ss"], textos["irpf"], None],
             "cells.values": [None, None, None, celdas]}
    caja = dict(x=0, y=1.0, xref="paper", yref="paper", xanchor="left", yanchor="bottom", yshift=8, align="left",
                showarrow=False, bgcolor="white", bordercolor=GRIS, borderwidth=1, borderpad=8,
                font=dict(size=13, color=TINTA),
                text=(f"<b>{NOMBRE_CCAA.get(ccaa, ccaa)} · {euros(s2024)} brutos en 2024</b> "
                      f"({euros(s2024 / 14)} al mes en 14 pagas)<br>"
                      f"IRPF de 2024: <b>{euros(u.real)}</b> ({pct(100 * u.real / s2024)} del bruto); "
                      f"con la tarifa actualizada, {euros(u.actualizada)}<br>"
                      f"De 2015 a 2024 habría pagado <b>{euros(d.real.sum() - d.actualizada.sum())} de más</b> "
                      f"por no actualizar la tarifa"))
    return datos, [caja]

def pasos(ccaa):
    lista = []
    for s in SUELDOS:
        datos, notas = estado(ccaa, s)
        lista.append(dict(method="update", label=f"{s:,.0f}".replace(",", "."),
                          args=[{k: v for k, v in datos.items()}, {"annotations": notas}]))
    return lista

PASOS = {c: pasos(c) for c in ipc.index}
datos0, notas0 = estado("Nacional", SUELDOS[INICIAL])

fig = make_subplots(rows=1, cols=2, column_widths=[0.52, 0.48], horizontal_spacing=0.06,
                    specs=[[{"type": "xy"}, {"type": "table"}]])
NOMBRE = {"neto": "Llega a la cuenta", "ss": "Seguridad Social", "irpf": "IRPF"}
for k, x, t in zip(("neto", "ss", "irpf"), datos0["x"], datos0["text"]):
    fig.add_trace(go.Bar(y=FILAS, x=x, orientation="h", name=NOMBRE[k], marker_color=COLOR[k], text=t,
                         textposition="inside", insidetextanchor="middle", textangle=0,
                         textfont=dict(color="white" if k == "irpf" else TINTA, size=12),
                         hovertemplate=NOMBRE[k] + ": <b>%{x:,.0f} €</b><extra></extra>"), row=1, col=1)
fig.add_trace(go.Table(
    columnwidth=[0.6, 1, 1, 1],
    header=dict(values=["<b>Año</b>", "<b>Sueldo</b>", "<b>Tarifa real</b>", "<b>Tarifa actualizada</b>"],
                fill_color="white", line_color=GRIS, align=["left", "right", "right", "right"],
                font=dict(color=TINTA, size=12), height=26),
    cells=dict(values=datos0["cells.values"][3], fill_color=[["white"] * 10 + ["#F1EFE8"]],
               line_color=GRIS, align=["left", "right", "right", "right"],
               font=dict(color=[TINTA2, TINTA2, NARANJA, AZUL], size=12), height=23)), row=1, col=2)

menu_ccaa = [dict(label=NOMBRE_CCAA.get(c, c), method="update",
                  args=[{k: v for k, v in estado(c, SUELDOS[INICIAL])[0].items()},
                        {"annotations": estado(c, SUELDOS[INICIAL])[1],
                         "sliders[0].steps": PASOS[c], "sliders[0].active": INICIAL}])
             for c in ipc.index]

fig.update_layout(
    barmode="stack", bargap=0.35, annotations=notas0,
    updatemenus=[dict(type="dropdown", buttons=menu_ccaa, active=0, x=0, y=1.33, xanchor="left", yanchor="bottom",
                      bgcolor="white", bordercolor=GRIS, font=dict(size=12, color=TINTA), showactive=True)],
    sliders=[dict(active=INICIAL, steps=PASOS["Nacional"], x=0, y=-0.10, len=1, pad=dict(t=30, b=0),
                  currentvalue=dict(prefix="Sueldo bruto de 2024: ", suffix=" €", font=dict(size=14, color=TINTA)),
                  ticklen=0, font=dict(size=1, color="white"), bgcolor=GRIS, activebgcolor=NARANJA,
                  bordercolor=GRIS, minorticklen=0)],
    xaxis=dict(range=[0, 100000], tickvals=[0, 25000, 50000, 75000, 100000],
               ticktext=["0", "25.000 €", "50.000 €", "75.000 €", "100.000 €"],
               gridcolor="#ECEAE3", zeroline=False, fixedrange=True),
    yaxis=dict(fixedrange=True, tickfont=dict(size=12, color=TINTA)),
    legend=dict(orientation="h", x=0.3, y=1.335, yanchor="bottom", traceorder="normal", font=dict(size=12)),
    font=dict(family="Inter, system-ui, -apple-system, Segoe UI, sans-serif", color=TINTA2, size=12),
    paper_bgcolor="white", plot_bgcolor="white", height=640, separators=",.",
    margin=dict(l=90, r=20, t=150, b=100),
)
fig.show(config={"displaylogo": False})

## 5. Guardar

- `figura8.html`: la calculadora, como página propia. Se coloca en StackBlitz junto a `index.html`, que la incrusta con un iframe. Pesa más que las otras figuras porque lleva calculadas todas las combinaciones de comunidad y sueldo.
- Para el PDF: el icono de la cámara, arriba a la derecha, descarga `figura8.png` con lo que se esté viendo. Súbelo a la carpeta `imagenes` del repositorio.

In [5]:
config = {"displaylogo": False, "responsive": True,
          "toImageButtonOptions": {"format": "png", "filename": "figura8", "scale": 2, "width": 1000, "height": 640}}
fig.write_html("figura8.html", include_plotlyjs="cdn", full_html=True, config=config)

# Misma tipografía que la web
html = open("figura8.html", encoding="utf-8").read()
fuente = ('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600&display=swap">'
          '<style>body{margin:0}</style>')
open("figura8.html", "w", encoding="utf-8").write(html.replace("<head>", "<head>" + fuente, 1))
print(f"Guardado: figura8.html ({os.path.getsize('figura8.html') / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download("figura8.html")
except ImportError:
    pass

Guardado: figura8.html (2.4 MB)
